# Imports

In [ ]:
import numpy as np
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import pandas as pd
import seaborn as sns
from jax import vmap
import jax.tree_util as jtu

from src.experiment import (
    ElectronSamplingExperiment,
    HeterogeneousReactionSamplingExperiment,
    AdsorptionSamplingExperiment,
)

from src.fdm import (
    AdsorptionReactionBackwardImplicitFDSolver,
    AdsorptionReactionExplicitFDSolver,
    AdsorptionReactionNewtonFDSolver,
    HeterogeneousReactionFDSolver,
    ElectronReactionFDSolver,
)

from src.params import (
    AdsorptionReactionParams,
    ElectronReactionParams,
    HeterogenousReactionParams,
)

from src.utils import generate_noisy_samples
from src.voltammetry import CyclicDC

sns.set_theme()
sns.set_context("paper", font_scale=1.5)


# Electron Transfer

$A \stackrel{E_f, K_0, \alpha}{\rightleftharpoons} B+e^{-}$

## Example Voltammagram

In [ ]:
voltammetry = CyclicDC()

fdm_solver = ElectronReactionFDSolver(voltammetry)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)

# Irreversible

irre_params = ElectronReactionParams(
    alpha=jnp.array(0.7), K0=jnp.array(1.0), Ef=jnp.array(0.0)
)

_, irre_current = fdm_solver.solve(irre_params)

ax1.plot(fdm_solver.applied_potentials, irre_current)
ax1.axhline(
    y=-0.496 * jnp.sqrt(irre_params.alpha) * jnp.sqrt(voltammetry.sigma),
    linestyle="--",
    c="red",
)
ax1.axvline(
    x=(jnp.log(irre_params.K0 / jnp.sqrt(irre_params.alpha * voltammetry.sigma)) - 0.78)
    / irre_params.alpha,
    linestyle="--",
    c="red",
)

rev_params = ElectronReactionParams(
    alpha=jnp.array(0.7), K0=jnp.array(100000.0), Ef=jnp.array(0.0)
)

_, rev_current = fdm_solver.solve(rev_params)

ax2.plot(fdm_solver.applied_potentials, rev_current)
ax2.axhline(y=-0.446 * jnp.sqrt(voltammetry.sigma), linestyle="--", c="red")
ax2.set_title("Reversible")
ax2.set_xlabel(r"$\theta$")

ax1.set_xlabel(r"$\theta$")
ax1.set_ylabel(r"$J$")
ax1.set_title("Irreversible")

plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## Effect of each parameter

In [ ]:
base_params = ElectronReactionParams(
    alpha=jnp.array(0.6), K0=jnp.array(10.0), Ef=jnp.array(0.5)
)

voltammetry = CyclicDC()

fd_solver = ElectronReactionFDSolver(voltammetry)

fig, (ax0, ax1, ax2) = plt.subplots(1, 3, figsize=(10, 3), sharex=True, sharey=True)

# alpha
alpha_range = jnp.array([0.3, 0.5, 0.7])
alpha_params = ElectronReactionParams(
    alpha=alpha_range,
    K0=jnp.full_like(alpha_range, base_params.K0),
    Ef=jnp.full_like(alpha_range, base_params.Ef),
)

_, alpha_currents = vmap(fd_solver.solve)(alpha_params)
for val, current in zip(alpha_range, alpha_currents):
    ax0.plot(fd_solver.applied_potentials, current, label=f"{val:.1f}")

# K0
K0_range = jnp.array([1.0, 10.0, 50.0])

K0_params = ElectronReactionParams(
    alpha=jnp.full_like(K0_range, base_params.alpha),
    K0=K0_range,
    Ef=jnp.full_like(K0_range, base_params.Ef),
)

_, K0_currents = vmap(fd_solver.solve)(K0_params)
for val, current in zip(K0_range, K0_currents):
    ax1.plot(fd_solver.applied_potentials, current, label=f"{val:.0f}")

# dB
Ef_range = jnp.array([-1.0, 0.0, 1.0])

Ef_params = ElectronReactionParams(
    alpha=jnp.full_like(Ef_range, base_params.alpha),
    K0=jnp.full_like(Ef_range, base_params.K0),
    Ef=Ef_range,
)

_, Ef_currents = vmap(fd_solver.solve)(Ef_params)
for val, current in zip(Ef_range, Ef_currents):
    ax2.plot(fd_solver.applied_potentials, current, label=f"{val:.1f}")


ax0.set_title(r"$\alpha$")
ax0.set_ylabel(r"$J$")
ax0.set_xlabel(r"$\theta$")
ax0.legend()

ax1.set_title(r"$K_0$")
ax1.set_xlabel(r"$\theta$")
ax1.legend()

ax2.set_title(r"$E_f$")
ax2.set_xlabel(r"$\theta$")
ax2.legend()

plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Noisy Sample Data

In [ ]:
params = ElectronReactionParams(
    alpha=jnp.array(0.6),
    K0=jnp.array(10.0),
    Ef=jnp.array(0.5),
)


plt.figure(figsize=(5, 4))
key = jr.key(0)

voltammetry = CyclicDC()
fd_solver = ElectronReactionFDSolver(voltammetry)
_, base_current = fd_solver.solve(params)

for sigma in [0.5, 0.25, 0.1]:
    noise_key, key = jr.split(key)
    noisy_current = generate_noisy_samples(1, base_current, sigma=sigma, key=noise_key)[
        0
    ]
    plt.plot(fd_solver.applied_potentials, noisy_current, label=sigma)


plt.ylabel(r"$J$")
plt.xlabel(r"$\theta$")

plt.legend()
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.show()

## Histogram Comparison

In [ ]:
hmc = np.load("./data/E_HMC_0.25_1000.npz")
rw = np.load("./data/E_RW_0.25_1000.npz")

voltammetry = CyclicDC()
fdm = ElectronReactionFDSolver(voltammetry)

true_params = ElectronSamplingExperiment().true_parameters
_, true_current = fdm_solver.solve(true_params)

alpha_est = jnp.mean(hmc["alpha"].flatten())
K0_est = jnp.mean(hmc["K0"].flatten())
Ef_est = jnp.mean(hmc["Ef"].flatten())
mean_params = ElectronReactionParams(alpha=alpha_est, K0=K0_est, Ef=Ef_est)

_, est_current = fdm_solver.solve(mean_params)


fig, (ax0, ax1, ax2, ax3) = plt.subplots(1, 4, figsize=(12, 3))

options = {"density": True, "bins": 50, "alpha": 0.8, "histtype": "step"}

ax0.set_title(r"$\alpha$")
ax0.set_ylabel("Density")
ax0.hist(hmc["alpha"].flatten(), **options, label="HMC")
ax0.hist(rw["alpha"].flatten(), **options, label="RW")
ax0.axvline(x=true_params.alpha, linestyle="--", color="black", label="True Value")

ax1.set_title(r"$K_0$")
ax1.hist(hmc["K0"].flatten(), **options)
ax1.hist(rw["K0"].flatten(), **options)
ax1.axvline(x=true_params.K0, linestyle="--", color="black")

ax2.set_title(r"$E_f$")
ax2.hist(hmc["Ef"].flatten(), **options)
ax2.hist(rw["Ef"].flatten(), **options)
ax2.axvline(x=true_params.Ef, linestyle="--", color="black")


ax3.set_title("Voltammagram")
ax3.plot(fdm_solver.applied_potentials, est_current)
ax3.plot(fdm_solver.applied_potentials, true_current, linestyle="--", color="black")
ax3.xaxis.set_inverted(True)
ax3.yaxis.set_inverted(True)

handles, labels = ax0.get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3)

plt.tight_layout(rect=[0, 0.1, 1, 1])
plt.show()

alpha_est, K0_est, Ef_est

## Biplot

In [ ]:
hmc = np.load("./data/E_HMC_0.25_1000.npz")
rw = np.load("./data/E_RW_0.25_1000.npz")

df_hmc = pd.DataFrame({k: hmc[k].flatten() for k in hmc.files})
df_rw = pd.DataFrame({k: rw[k].flatten() for k in rw.files})

df_hmc["sampler"] = "HMC"
df_rw["sampler"] = "RW"

df = pd.concat([df_hmc, df_rw], ignore_index=True)

label_map = {
    "alpha": r"$\alpha$",
    "K0": r"$K_0$",
    "Ef": r"$E_f$",
}

cols = [c for c in hmc.files if c != "logdensity"]
df_plot = df.drop(columns=["logdensity"]).rename(columns=label_map)
plot_cols = [label_map[c] for c in cols]

g = sns.PairGrid(
    df_plot,
    vars=plot_cols,
    hue="sampler",
    hue_order=["HMC", "RW"],
    diag_sharey=False,
    layout_pad=True,
)


# Diagonal: normalised histograms for both samplers
def diag_hist(x, **kwargs):
    ax = plt.gca()
    ax.hist(x, density=True, bins=50, histtype="step")


g.map_diag(diag_hist)


def lower_kde(x, y, **kwargs):
    label = kwargs.pop("label", None)
    if label == "HMC":
        sns.kdeplot(x=x, y=y, ax=plt.gca(), **kwargs)


g.map_lower(lower_kde, label=g._hue_var)


def upper_kde(x, y, **kwargs):
    label = kwargs.pop("label", None)
    if label == "RW":
        sns.kdeplot(x=x, y=y, ax=plt.gca(), **kwargs)


g.map_upper(upper_kde, label=g._hue_var)

legend_handles = [
    mpatches.Patch(color="C0", label="HMC"),
    mpatches.Patch(color="C1", label="RW"),
]
g.figure.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=2,
    frameon=True,
)

plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.show()


## Effect of noise

In [ ]:
hmc = np.load("./data/E_HMC_0.50_1000.npz")
rw = np.load("./data/E_RW_0.50_1000.npz")

voltammetry = CyclicDC()
fdm = ElectronReactionFDSolver(voltammetry)

true_params = ElectronSamplingExperiment().true_parameters
_, true_current = fdm_solver.solve(true_params)

alpha_est = jnp.mean(hmc["alpha"].flatten())
K0_est = jnp.mean(hmc["K0"].flatten())
Ef_est = jnp.mean(hmc["Ef"].flatten())
print(alpha_est, K0_est, Ef_est)
mean_params = ElectronReactionParams(alpha=alpha_est, K0=K0_est, Ef=Ef_est)

_, est_current = fdm_solver.solve(mean_params)


fig, (ax0, ax1, ax2, ax3) = plt.subplots(1, 4, figsize=(12, 3))

options = {"density": True, "bins": 50, "alpha": 0.8, "histtype": "step"}

ax0.set_title(r"$\alpha$")
ax0.set_ylabel("Density")
ax0.hist(hmc["alpha"].flatten(), **options, label="HMC")
ax0.hist(rw["alpha"].flatten(), **options, label="RW")
ax0.axvline(x=true_params.alpha, linestyle="--", color="black", label="True Value")

ax1.set_title(r"$K_0$")
ax1.hist(hmc["K0"].flatten(), **options)
ax1.hist(rw["K0"].flatten(), **options)
ax1.axvline(x=true_params.K0, linestyle="--", color="black")

ax2.set_title(r"$E_f$")
ax2.hist(hmc["Ef"].flatten(), **options)
ax2.hist(rw["Ef"].flatten(), **options)
ax2.axvline(x=true_params.Ef, linestyle="--", color="black")

ax3.set_title("Voltammagram")
ax3.plot(fdm_solver.applied_potentials, est_current)
ax3.plot(fdm_solver.applied_potentials, true_current, linestyle="--", color="black")
ax3.xaxis.set_inverted(True)
ax3.yaxis.set_inverted(True)


handles, labels = ax0.get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3)

plt.tight_layout(rect=[0, 0.1, 1, 1])
plt.show()


## Path of moments

In [ ]:
def compute_cum_moments(x):
    n = jnp.arange(1, len(x) + 1)
    cum_mean = jnp.cumsum(x) / n
    cum_sq_mean = jnp.cumsum(x**2) / n
    variance = jnp.maximum(cum_sq_mean - cum_mean**2, 0)
    return cum_mean, jnp.sqrt(variance)


hmc = np.load("./data/E_HMC_0.25_1000.npz")
rw = np.load("./data/E_RW_0.25_1000.npz")

hmc_K0 = hmc["K0"].flatten()
rw_K0 = rw["K0"].flatten()


def plot_moment(key: str, ax):

    hmc_param = hmc[key].flatten()
    rw_param = rw[key].flatten()

    hmc_cum_mean, hmc_cum_std = compute_cum_moments(hmc_param)
    rw_cum_mean, rw_cum_std = compute_cum_moments(rw_param)

    hmc_x = np.arange(len(hmc_cum_mean)) / len(hmc_cum_mean)
    rw_x = np.arange(len(rw_cum_mean)) / len(rw_cum_mean)

    ax.plot(hmc_x, hmc_cum_mean, label="HMC")
    ax.fill_between(
        hmc_x,
        hmc_cum_mean + hmc_cum_std,
        hmc_cum_mean - hmc_cum_std,
        alpha=0.5,
    )

    ax.plot(rw_x, rw_cum_mean, label="RW")
    ax.fill_between(
        rw_x,
        rw_cum_mean + rw_cum_std,
        rw_cum_mean - rw_cum_std,
        alpha=0.5,
    )


fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(10, 4), sharex=True)
plot_moment("alpha", ax1)
plot_moment("K0", ax2)
plot_moment("Ef", ax3)

ax1.set_title(r"$\alpha$")
ax2.set_title(r"$K_0$")
ax3.set_title(r"$E_f$")

handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2)

plt.tight_layout(rect=[0, 0.075, 1, 1])
plt.show()

# Heterogeneous
$
\begin{gathered}
\mathrm{A}+\mathrm{e}^{-} \rightleftarrows \mathrm{B} \\
\mathrm{~B} \xrightarrow{k_{\text {het }}} \mathrm{C} \\
\mathrm{C}+\mathrm{e}^{-} \rightleftarrows \mathrm{D}
\end{gathered}
$

## Heterogeneous Parameter Effects

In [ ]:
voltammetry = CyclicDC()
fd_solver = HeterogeneousReactionFDSolver(voltammetry)

base_params = HeterogenousReactionParams(
    alpha_1=jnp.array(0.6),
    K0_1=jnp.array(10.0),
    Ef_1=jnp.array(0.5),
    alpha_2=jnp.array(0.6),
    K0_2=jnp.array(5.0),
    Ef_2=jnp.array(0.2),
    K_het=jnp.array(20.0),
)

fig = plt.figure(figsize=(12, 6))

gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.4, wspace=0.3)

ax_a1 = fig.add_subplot(gs[0, 0])
ax_K1 = fig.add_subplot(gs[0, 1])
ax_Ef1 = fig.add_subplot(gs[0, 2])
ax_a2 = fig.add_subplot(gs[1, 0])
ax_K2 = fig.add_subplot(gs[1, 1])
ax_Ef2 = fig.add_subplot(gs[1, 2])

gs_right = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=gs[:, 3], hspace=0)
ax_Khet = fig.add_subplot(gs_right[1, 0])

all_axes = [ax_a1, ax_K1, ax_Ef1, ax_a2, ax_K2, ax_Ef2, ax_Khet]
for ax in all_axes[1:]:
    ax.sharex(all_axes[0])
    ax.sharey(all_axes[0])

# alpha_1
alpha_1_range = jnp.array([0.3, 0.5, 0.7])
alpha_1_params = HeterogenousReactionParams(
    alpha_1=alpha_1_range,
    K0_1=jnp.full_like(alpha_1_range, base_params.K0_1),
    Ef_1=jnp.full_like(alpha_1_range, base_params.Ef_1),
    alpha_2=jnp.full_like(alpha_1_range, base_params.alpha_2),
    K0_2=jnp.full_like(alpha_1_range, base_params.K0_1),
    Ef_2=jnp.full_like(alpha_1_range, base_params.Ef_1),
    K_het=jnp.full_like(alpha_1_range, base_params.K_het),
)

_, alpha_1_currents = vmap(fd_solver.solve)(alpha_1_params)

for val, current in zip(alpha_1_range, alpha_1_currents):
    ax_a1.plot(fd_solver.applied_potentials, current, label=f"{val:.1f}")

ax_a1.legend()
ax_a1.set_title(r"$\alpha^{(1)}$")
ax_a1.set_ylabel(r"$J$")

# alpha_2
alpha_2_range = jnp.array([0.3, 0.5, 0.7])
alpha_2_params = HeterogenousReactionParams(
    alpha_1=jnp.full_like(alpha_2_range, base_params.alpha_1),
    K0_1=jnp.full_like(alpha_2_range, base_params.K0_1),
    Ef_1=jnp.full_like(alpha_2_range, base_params.Ef_1),
    alpha_2=alpha_2_range,
    K0_2=jnp.full_like(alpha_2_range, base_params.K0_1),
    Ef_2=jnp.full_like(alpha_2_range, base_params.Ef_1),
    K_het=jnp.full_like(alpha_2_range, base_params.K_het),
)

_, alpha_2_currents = vmap(fd_solver.solve)(alpha_2_params)

for val, current in zip(alpha_2_range, alpha_2_currents):
    ax_a2.plot(fd_solver.applied_potentials, current, label=f"{val:.1f}")

ax_a2.legend()
ax_a2.set_title(r"$\alpha^{(2)}$")
ax_a2.set_ylabel(r"$J$")

# K0_1
K0_1_range = jnp.array([1.0, 5.0, 20.0])
K0_1_params = HeterogenousReactionParams(
    alpha_1=jnp.full_like(K0_1_range, base_params.alpha_1),
    K0_1=K0_1_range,
    Ef_1=jnp.full_like(K0_1_range, base_params.Ef_1),
    alpha_2=jnp.full_like(K0_1_range, base_params.alpha_2),
    K0_2=jnp.full_like(K0_1_range, base_params.K0_2),
    Ef_2=jnp.full_like(K0_1_range, base_params.Ef_2),
    K_het=jnp.full_like(K0_1_range, base_params.K_het),
)

_, K0_1_currents = vmap(fd_solver.solve)(K0_1_params)

for val, current in zip(K0_1_range, K0_1_currents):
    ax_K1.plot(fd_solver.applied_potentials, current, label=f"{val:.0f}")

ax_K1.legend()
ax_K1.set_title(r"$K_0^{(1)}$")

# K0_2
K0_2_range = jnp.array([1.0, 5.0, 20.0])
K0_2_params = HeterogenousReactionParams(
    alpha_1=jnp.full_like(K0_2_range, base_params.alpha_1),
    K0_1=jnp.full_like(K0_2_range, base_params.K0_1),
    Ef_1=jnp.full_like(K0_2_range, base_params.Ef_1),
    alpha_2=jnp.full_like(K0_2_range, base_params.alpha_2),
    K0_2=K0_2_range,
    Ef_2=jnp.full_like(K0_2_range, base_params.Ef_2),
    K_het=jnp.full_like(K0_2_range, base_params.K_het),
)

_, K0_2_currents = vmap(fd_solver.solve)(K0_2_params)

for val, current in zip(K0_2_range, K0_2_currents):
    ax_K2.plot(fd_solver.applied_potentials, current, label=f"{val:.0f}")

ax_K2.legend()
ax_K2.set_title(r"$K_0^{(2)}$")

# Ef_1
Ef_1_range = jnp.array([-1.0, 0.0, 1.0])
Ef_1_params = HeterogenousReactionParams(
    alpha_1=jnp.full_like(Ef_1_range, base_params.alpha_1),
    K0_1=jnp.full_like(Ef_1_range, base_params.K0_1),
    Ef_1=Ef_1_range,
    alpha_2=jnp.full_like(Ef_1_range, base_params.alpha_2),
    K0_2=jnp.full_like(Ef_1_range, base_params.K0_2),
    Ef_2=jnp.full_like(Ef_1_range, base_params.Ef_2),
    K_het=jnp.full_like(Ef_1_range, base_params.K_het),
)

_, Ef_1_currents = vmap(fd_solver.solve)(Ef_1_params)

for val, current in zip(Ef_1_range, Ef_1_currents):
    ax_Ef1.plot(fd_solver.applied_potentials, current, label=f"{val:.0f}")

ax_Ef1.legend()
ax_Ef1.set_title(r"$E_f^{(1)}$")

# Ef_2
Ef_2_range = jnp.array([-1.0, 0.0, 1.0])
Ef_2_params = HeterogenousReactionParams(
    alpha_1=jnp.full_like(Ef_2_range, base_params.alpha_1),
    K0_1=jnp.full_like(Ef_2_range, base_params.K0_1),
    Ef_1=jnp.full_like(Ef_2_range, base_params.Ef_1),
    alpha_2=jnp.full_like(Ef_2_range, base_params.alpha_2),
    K0_2=jnp.full_like(Ef_2_range, base_params.K0_2),
    Ef_2=Ef_2_range,
    K_het=jnp.full_like(Ef_2_range, base_params.K_het),
)

_, Ef_2_currents = vmap(fd_solver.solve)(Ef_2_params)

for val, current in zip(Ef_2_range, Ef_2_currents):
    ax_Ef2.plot(fd_solver.applied_potentials, current, label=f"{val:.0f}")

ax_Ef2.legend()
ax_Ef2.set_title(r"$E_f^{(2)}$")

# Khet
Khet_range = jnp.array([1.0, 5.0, 20.0])
Khet_params = HeterogenousReactionParams(
    alpha_1=jnp.full_like(Khet_range, base_params.alpha_1),
    K0_1=jnp.full_like(Khet_range, base_params.K0_1),
    Ef_1=jnp.full_like(Khet_range, base_params.Ef_1),
    alpha_2=jnp.full_like(Khet_range, base_params.alpha_2),
    K0_2=jnp.full_like(Khet_range, base_params.K0_2),
    Ef_2=jnp.full_like(Khet_range, base_params.Ef_1),
    K_het=Khet_range,
)

_, Khet_currents = vmap(fd_solver.solve)(Khet_params)

for val, current in zip(Khet_range, Khet_currents):
    ax_Khet.plot(fd_solver.applied_potentials, current, label=f"{val:.0f}")

ax_Khet.legend()
ax_Khet.set_title(r"$K_{het}$")

plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.show()

## Flux contributions

In [ ]:
voltammetry = CyclicDC()
fd_solver = HeterogeneousReactionFDSolver(voltammetry)

params = HeterogenousReactionParams(
    alpha_1=jnp.array(0.6),
    K0_1=jnp.array(10.0),
    Ef_1=jnp.array(0.5),
    alpha_2=jnp.array(0.4),
    K0_2=jnp.array(5.0),
    Ef_2=jnp.array(0.2),
    K_het=jnp.array(20.0),
)


def current_contributions(c):
    c0_A = c[2 * fd_solver.Nx - 2]
    c1_A = c[2 * fd_solver.Nx - 4]
    c2_A = c[2 * fd_solver.Nx - 6]

    c0_C = c[2 * fd_solver.Nx]
    c1_C = c[2 * fd_solver.Nx + 2]
    c2_C = c[2 * fd_solver.Nx + 4]

    h1 = fd_solver.X[1] - fd_solver.X[0]
    h2 = fd_solver.X[2] - fd_solver.X[0]

    dcA_dx = (h2**2 * (c0_A - c1_A) + h1**2 * (c2_A - c0_A)) / (h1 * h2 * (h1 - h2))
    dcC_dx = (h2**2 * (c0_C - c1_C) + h1**2 * (c2_C - c0_C)) / (h1 * h2 * (h1 - h2))

    K_het_cB = params.K_het * c[2 * fd_solver.Nx - 1]

    return -dcA_dx, -dcC_dx, -K_het_cB


solution, current = fd_solver.solve(params)
CA, CC, KhetB = vmap(current_contributions)(solution)

plt.plot(fd_solver.applied_potentials, current, label="Total")
plt.plot(fd_solver.applied_potentials, CA, label=r"$\frac{\partial C_A}{\partial X}$")
plt.plot(fd_solver.applied_potentials, CC, label=r"$\frac{\partial C_C}{\partial X}$")
plt.plot(fd_solver.applied_potentials, KhetB, label=r"$K_{\text{het}}C_B$")
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.xlabel(r"$\theta$")
plt.ylabel(r"$J$")
plt.legend()
plt.show()

## Histogram 

In [ ]:
hmc = np.load("./data/H_HMC_0.25_1000.npz")
rw = np.load("./data/H_RW_0.25_1000.npz")

true_params = HeterogeneousReactionSamplingExperiment().true_parameters

fig = plt.figure(figsize=(12, 5))

gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.4, wspace=0.3)

ax_a1 = fig.add_subplot(gs[0, 0])
ax_K1 = fig.add_subplot(gs[0, 1])
ax_Ef1 = fig.add_subplot(gs[0, 2])
ax_a2 = fig.add_subplot(gs[1, 0])
ax_K2 = fig.add_subplot(gs[1, 1])
ax_Ef2 = fig.add_subplot(gs[1, 2])

gs_right = gridspec.GridSpecFromSubplotSpec(3, 1, subplot_spec=gs[:, 3], hspace=0)
ax_Khet = fig.add_subplot(gs_right[1, 0])

options = {"density": True, "bins": 50, "alpha": 0.8, "histtype": "step"}

ax_a1.set_title(r"$\alpha^{(1)}$")
ax_a1.hist(hmc["alpha_1"].flatten(), **options, label="HMC")
ax_a1.hist(rw["alpha_1"].flatten(), **options, label="RW")
ax_a1.axvline(x=true_params.alpha_1, linestyle="--", color="black", label="True Value")

ax_a2.set_title(r"$\alpha^{(2)}$")
ax_a2.hist(hmc["alpha_2"].flatten(), **options)
ax_a2.hist(rw["alpha_2"].flatten(), **options)
ax_a2.axvline(x=true_params.alpha_2, linestyle="--", color="black")

ax_K1.set_title(r"$K_0^{(1)}$")
ax_K1.hist(hmc["K0_1"].flatten(), **options)
ax_K1.hist(rw["K0_1"].flatten(), **options)
ax_K1.axvline(x=true_params.K0_1, linestyle="--", color="black")

ax_K2.set_title(r"$K_0^{(2)}$")
ax_K2.hist(hmc["K0_2"].flatten(), **options)
ax_K2.hist(rw["K0_2"].flatten(), **options)
ax_K2.axvline(x=true_params.K0_2, linestyle="--", color="black")

ax_Ef1.set_title(r"$E_f^{(1)}$")
ax_Ef1.hist(hmc["Ef_1"].flatten(), **options)
ax_Ef1.hist(rw["Ef_1"].flatten(), **options)
ax_Ef1.axvline(x=true_params.Ef_1, linestyle="--", color="black")

ax_Ef2.set_title(r"$E_f^{(2)}$")
ax_Ef2.hist(hmc["Ef_2"].flatten(), **options)
ax_Ef2.hist(rw["Ef_2"].flatten(), **options)
ax_Ef2.axvline(x=true_params.Ef_2, linestyle="--", color="black")

ax_Khet.set_title(r"$K_{\text{het}}$")
ax_Khet.hist(hmc["K_het"].flatten(), **options)
ax_Khet.hist(rw["K_het"].flatten(), **options)
ax_Khet.axvline(x=true_params.K_het, linestyle="--", color="black")

handles, labels = ax_a1.get_legend_handles_labels()
fig.legend(handles, labels, loc="lower right", ncol=1)

plt.show()

## Biplot

In [ ]:
rw = np.load("./data/H_RW_0.25_1000.npz")

df = pd.DataFrame({k: rw[k].flatten() for k in rw.files})

label_map = {
    "alpha_1": r"$\alpha^{(1)}$",
    "alpha_2": r"$\alpha^{(2)}$",
    "K0_1": r"$K_0^{(1)}$",
    "K0_2": r"$K_0^{(2)}$",
    "Ef_1": r"$E_f^{(1)}$",
    "Ef_2": r"$E_f^{(2)}$",
    "K_het": r"$K_{\text{het}}$",
}

cols = [c for c in rw.files if c != "logdensity"]
df_plot = df.drop(columns=["logdensity"]).rename(columns=label_map)
plot_cols = [label_map[c] for c in cols]

g = sns.PairGrid(
    df_plot, vars=plot_cols, diag_sharey=False, layout_pad=True, corner=True
)


def diag_hist(x, **kwargs):
    ax = plt.gca()
    ax.hist(x, density=True, bins=50, histtype="step")


g.map_diag(diag_hist)


def lower_kde(x, y, **kwargs):
    sns.kdeplot(x=x, y=y, ax=plt.gca(), **kwargs)


g.map_lower(lower_kde, label=g._hue_var)


plt.tight_layout()
plt.show()


# Absorption

## Adsorption Example Voltammagrams

In [ ]:
voltammetry = CyclicDC(sigma=20, theta_i=20, theta_v=-20)
fdm_solver = AdsorptionReactionNewtonFDSolver(voltammetry, h0=1e-5, dtheta=2e-1)

fig, axs = plt.subplots(2, 2, figsize=(8, 5), sharex=True)

params = AdsorptionReactionParams(
    alpha_sol=jnp.array(0.5),
    K0_sol=jnp.array(0.0),
    Ef_sol=jnp.array(0.0),
    alpha_ads=jnp.array(0.5),
    K0_ads=jnp.array(1e6),
    K_A_ads=jnp.array(1e3),
    K_A_des=jnp.array(1e-3),
    K_B_ads=jnp.array(1.0),
    K_B_des=jnp.array(1e-3),
)

_, current = fdm_solver.solve(params)

axs[0, 0].set_title("Monolayer")
axs[0, 0].plot(fdm_solver.applied_potentials, current)
axs[0, 0].yaxis.set_inverted(True)
axs[0, 0].set_ylabel(r"$J$")

params = AdsorptionReactionParams(
    alpha_sol=jnp.array(0.4),
    K0_sol=jnp.array(1e-3),
    Ef_sol=jnp.array(0.0),
    alpha_ads=jnp.array(0.45),
    K0_ads=jnp.array(1.0),
    K_A_ads=jnp.array(4.5),
    K_A_des=jnp.array(0.5),
    K_B_ads=jnp.array(0.5),
    K_B_des=jnp.array(5.0),
)

_, current = fdm_solver.solve(params)

axs[0, 1].set_title("Reactant absorbed more strongly")
axs[0, 1].plot(fdm_solver.applied_potentials, current)
axs[0, 1].yaxis.set_inverted(True)

params = AdsorptionReactionParams(
    alpha_sol=jnp.array(0.9),
    K0_sol=jnp.array(1000.0),
    Ef_sol=jnp.array(0.0),
    alpha_ads=jnp.array(0.9),
    K0_ads=jnp.array(0.0),
    K_A_ads=jnp.array(1.0),
    K_A_des=jnp.array(1.0),
    K_B_ads=jnp.array(1.0),
    K_B_des=jnp.array(1.0),
)

_, current = fdm_solver.solve(params)

axs[1, 0].set_title("Diffusional")
axs[1, 0].plot(fdm_solver.applied_potentials, current)
axs[1, 0].yaxis.set_inverted(True)
axs[1, 0].set_ylabel(r"$J$")
axs[1, 0].set_xlabel(r"$\theta$")

params = AdsorptionReactionParams(
    alpha_sol=jnp.array(0.4),
    K0_sol=jnp.array(1.0),
    Ef_sol=jnp.array(0.0),
    alpha_ads=jnp.array(0.45),
    K0_ads=jnp.array(5e-1),
    K_A_ads=jnp.array(0.1),
    K_A_des=jnp.array(5.0),
    K_B_ads=jnp.array(5.0),
    K_B_des=jnp.array(1e-1),
)

_, current = fdm_solver.solve(params)

axs[1, 1].set_title("Product absorbed more strongly")
axs[1, 1].plot(fdm_solver.applied_potentials, current)
axs[1, 1].yaxis.set_inverted(True)

plt.gca().invert_xaxis()
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()


## Parameter Effect

In [ ]:
voltammetry = CyclicDC(sigma=1, theta_i=20, theta_v=-20)
fdm_solver = AdsorptionReactionNewtonFDSolver(voltammetry)

fig, axs = plt.subplots(3, 3, figsize=(10, 8), sharex=True, sharey=True)

params = AdsorptionReactionParams(
    alpha_sol=jnp.array(0.4),
    K0_sol=jnp.array(1e-3),
    Ef_sol=jnp.array(0.0),
    alpha_ads=jnp.array(0.45),
    K0_ads=jnp.array(5e-1),
    K_A_ads=jnp.array(4.5),
    K_A_des=jnp.array(1.0),
    K_B_ads=jnp.array(1.0),
    K_B_des=jnp.array(1.0),
)

# Alpha Solution

alpha_sol_range = jnp.array([0.3, 0.5, 0.7])
alpha_sol_param = AdsorptionReactionParams(
    alpha_sol=alpha_sol_range,
    K0_sol=jnp.full_like(alpha_sol_range, params.K0_sol),
    Ef_sol=jnp.full_like(alpha_sol_range, params.Ef_sol),
    alpha_ads=jnp.full_like(alpha_sol_range, params.alpha_ads),
    K0_ads=jnp.full_like(alpha_sol_range, params.K0_ads),
    K_A_ads=jnp.full_like(alpha_sol_range, params.K_A_ads),
    K_A_des=jnp.full_like(alpha_sol_range, params.K_A_des),
    K_B_ads=jnp.full_like(alpha_sol_range, params.K_B_ads),
    K_B_des=jnp.full_like(alpha_sol_range, params.K_B_des),
)
_, alpha_sol_currents = vmap(fdm_solver.solve)(alpha_sol_param)

for current, val in zip(alpha_sol_currents, alpha_sol_range):
    axs[0, 0].plot(fdm_solver.applied_potentials, current, label=f"{val:.1f}")

axs[0, 0].legend(fontsize="small")
axs[0, 0].set_title(r"$\alpha^{(sol)}$")

# Alpha Adsorption
alpha_ads_range = jnp.array([0.3, 0.5, 0.7])
alpha_ads_param = AdsorptionReactionParams(
    alpha_sol=jnp.full_like(alpha_ads_range, params.alpha_sol),
    K0_sol=jnp.full_like(alpha_ads_range, params.K0_sol),
    Ef_sol=jnp.full_like(alpha_ads_range, params.Ef_sol),
    alpha_ads=alpha_ads_range,
    K0_ads=jnp.full_like(alpha_ads_range, params.K0_ads),
    K_A_ads=jnp.full_like(alpha_ads_range, params.K_A_ads),
    K_A_des=jnp.full_like(alpha_ads_range, params.K_A_des),
    K_B_ads=jnp.full_like(alpha_ads_range, params.K_B_ads),
    K_B_des=jnp.full_like(alpha_ads_range, params.K_B_des),
)
_, alpha_ads_currents = vmap(fdm_solver.solve)(alpha_ads_param)

for current, val in zip(alpha_ads_currents, alpha_ads_range):
    axs[1, 0].plot(fdm_solver.applied_potentials, current, label=f"{val:.1f}")

axs[1, 0].legend(fontsize="small")
axs[1, 0].set_title(r"$\alpha^{(ads)}$")

# K0 Solution
K0_sol_range = jnp.array([1e-2, 0.1, 1.0])
K0_sol_param = AdsorptionReactionParams(
    alpha_sol=jnp.full_like(K0_sol_range, params.alpha_sol),
    K0_sol=K0_sol_range,
    Ef_sol=jnp.full_like(K0_sol_range, params.Ef_sol),
    alpha_ads=jnp.full_like(K0_sol_range, params.alpha_ads),
    K0_ads=jnp.full_like(K0_sol_range, params.K0_ads),
    K_A_ads=jnp.full_like(K0_sol_range, params.K_A_ads),
    K_A_des=jnp.full_like(K0_sol_range, params.K_A_des),
    K_B_ads=jnp.full_like(K0_sol_range, params.K_B_ads),
    K_B_des=jnp.full_like(K0_sol_range, params.K_B_des),
)
_, K0_sol_currents = vmap(fdm_solver.solve)(K0_sol_param)

for current, val in zip(K0_sol_currents, K0_sol_range):
    axs[0, 1].plot(fdm_solver.applied_potentials, current, label=f"{val:.2f}")

axs[0, 1].legend(fontsize="small")
axs[0, 1].set_title(r"$K_0^{(sol)}$")

# K0 Adsorption
K0_ads_range = jnp.array([1e-2, 0.1, 1.0])
K0_ads_param = AdsorptionReactionParams(
    alpha_sol=jnp.full_like(K0_ads_range, params.alpha_sol),
    K0_sol=jnp.full_like(K0_ads_range, params.K0_sol),
    Ef_sol=jnp.full_like(K0_ads_range, params.Ef_sol),
    alpha_ads=jnp.full_like(K0_ads_range, params.alpha_ads),
    K0_ads=K0_ads_range,
    K_A_ads=jnp.full_like(K0_ads_range, params.K_A_ads),
    K_A_des=jnp.full_like(K0_ads_range, params.K_A_des),
    K_B_ads=jnp.full_like(K0_ads_range, params.K_B_ads),
    K_B_des=jnp.full_like(K0_ads_range, params.K_B_des),
)
_, K0_ads_currents = vmap(fdm_solver.solve)(K0_ads_param)

for current, val in zip(K0_ads_currents, K0_ads_range):
    axs[1, 1].plot(fdm_solver.applied_potentials, current, label=f"{val:.2f}")

axs[1, 1].legend(fontsize="small")
axs[1, 1].set_title(r"$K_0^{(ads)}$")


# Ef Solution
Ef_sol_range = jnp.array([-1, 0.0, 1])
Ef_sol_param = AdsorptionReactionParams(
    alpha_sol=jnp.full_like(Ef_sol_range, params.alpha_sol),
    K0_sol=jnp.full_like(Ef_sol_range, params.K0_sol),
    Ef_sol=Ef_sol_range,
    alpha_ads=jnp.full_like(Ef_sol_range, params.alpha_ads),
    K0_ads=jnp.full_like(Ef_sol_range, params.K0_ads),
    K_A_ads=jnp.full_like(Ef_sol_range, params.K_A_ads),
    K_A_des=jnp.full_like(Ef_sol_range, params.K_A_des),
    K_B_ads=jnp.full_like(Ef_sol_range, params.K_B_ads),
    K_B_des=jnp.full_like(Ef_sol_range, params.K_B_des),
)
_, Ef_sol_currents = vmap(fdm_solver.solve)(Ef_sol_param)

for current, val in zip(Ef_sol_currents, Ef_sol_range):
    axs[0, 2].plot(fdm_solver.applied_potentials, current, label=f"{val:.2f}")

axs[0, 2].legend(fontsize="small")
axs[0, 2].set_title(r"$E_f^{(ads)}$")

# K_A_ads Solution
K_A_ads_range = jnp.array([1.0, 5.0, 10.0])
K_A_ads_param = AdsorptionReactionParams(
    alpha_sol=jnp.full_like(K_A_ads_range, params.alpha_sol),
    K0_sol=jnp.full_like(K_A_ads_range, params.K0_sol),
    Ef_sol=jnp.full_like(K_A_ads_range, params.Ef_sol),
    alpha_ads=jnp.full_like(K_A_ads_range, params.alpha_ads),
    K0_ads=jnp.full_like(K_A_ads_range, params.K0_ads),
    K_A_ads=K_A_ads_range,
    K_A_des=jnp.full_like(K_A_ads_range, params.K_A_des),
    K_B_ads=jnp.full_like(K_A_ads_range, params.K_B_ads),
    K_B_des=jnp.full_like(K_A_ads_range, params.K_B_des),
)
_, K_A_ads_currents = vmap(fdm_solver.solve)(K_A_ads_param)

for current, val in zip(K_A_ads_currents, K_A_ads_range):
    axs[1, 2].plot(fdm_solver.applied_potentials, current, label=f"{val:.1f}")

axs[1, 2].legend(fontsize="small")
axs[1, 2].set_title(r"$K_{ads}^A$")

# K_A_des Solution
K_A_des_range = jnp.array([0.1, 0.5, 1.0])
K_A_des_param = AdsorptionReactionParams(
    alpha_sol=jnp.full_like(K_A_des_range, params.alpha_sol),
    K0_sol=jnp.full_like(K_A_des_range, params.K0_sol),
    Ef_sol=jnp.full_like(K_A_des_range, params.K_A_des),
    alpha_ads=jnp.full_like(K_A_des_range, params.alpha_ads),
    K0_ads=jnp.full_like(K_A_des_range, params.K0_ads),
    K_A_ads=jnp.full_like(K_A_des_range, params.K_A_ads),
    K_A_des=K_A_des_range,
    K_B_ads=jnp.full_like(K_A_des_range, params.K_B_ads),
    K_B_des=jnp.full_like(K_A_des_range, params.K_B_des),
)
_, K_A_des_currents = vmap(fdm_solver.solve)(K_A_des_param)

for current, val in zip(K_A_des_currents, K_A_des_range):
    axs[2, 0].plot(fdm_solver.applied_potentials, current, label=f"{val:.1f}")

axs[2, 0].legend(fontsize="small")
axs[2, 0].set_title(r"$K_{des}^A$")

# K_B_ads Solution
K_B_ads_range = jnp.array([0.1, 0.5, 1.0])
K_B_ads_param = AdsorptionReactionParams(
    alpha_sol=jnp.full_like(K_B_ads_range, params.alpha_sol),
    K0_sol=jnp.full_like(K_B_ads_range, params.K0_sol),
    Ef_sol=jnp.full_like(K_B_ads_range, params.Ef_sol),
    alpha_ads=jnp.full_like(K_B_ads_range, params.alpha_ads),
    K0_ads=jnp.full_like(K_B_ads_range, params.K0_ads),
    K_A_ads=jnp.full_like(K_B_ads_range, params.K_A_ads),
    K_A_des=jnp.full_like(K_B_ads_range, params.K_A_des),
    K_B_ads=K_B_ads_range,
    K_B_des=jnp.full_like(K_B_ads_range, params.K_B_des),
)
_, K_B_ads_currents = vmap(fdm_solver.solve)(K_B_ads_param)

for current, val in zip(K_B_ads_currents, K_B_ads_range):
    axs[2, 1].plot(fdm_solver.applied_potentials, current, label=f"{val:.1f}")

axs[2, 1].legend(fontsize="small")
axs[2, 1].set_title(r"$K_{ads}^B$")

# K_B_des Solution
K_B_des_range = jnp.array([0.1, 0.5, 1.0])
K_B_des_param = AdsorptionReactionParams(
    alpha_sol=jnp.full_like(K_B_des_range, params.alpha_sol),
    K0_sol=jnp.full_like(K_B_des_range, params.K0_sol),
    Ef_sol=jnp.full_like(K_B_des_range, params.K_A_des),
    alpha_ads=jnp.full_like(K_B_des_range, params.alpha_ads),
    K0_ads=jnp.full_like(K_B_des_range, params.K0_ads),
    K_A_ads=jnp.full_like(K_B_des_range, params.K_A_ads),
    K_A_des=jnp.full_like(K_B_des_range, params.K_A_des),
    K_B_ads=jnp.full_like(K_A_des_range, params.K_B_ads),
    K_B_des=K_B_des_range,
)
_, K_B_des_currents = vmap(fdm_solver.solve)(K_B_des_param)

for current, val in zip(K_B_des_currents, K_B_des_range):
    axs[2, 2].plot(fdm_solver.applied_potentials, current, label=f"{val:.1f}")

axs[2, 2].legend(fontsize="small")
axs[2, 2].set_title(r"$K_{des}^B$")

plt.gca().invert_xaxis()
plt.gca().invert_yaxis()
plt.subplots_adjust(wspace=0.02)
plt.show()

## Numerics Comparison for Adsorption

In [ ]:
voltammetry = CyclicDC(sigma=10, theta_i=25, theta_v=-25)
fd_newton = AdsorptionReactionNewtonFDSolver(voltammetry, h0=1e-5, dtheta=2e-1)
fd_explicit = AdsorptionReactionExplicitFDSolver(voltammetry, h0=1e-5, dtheta=2e-1)
fd_backward_implicit = AdsorptionReactionBackwardImplicitFDSolver(
    voltammetry, h0=1e-5, dtheta=2e-1
)

params = AdsorptionReactionParams(
    alpha_sol=jnp.array(0.4),
    K0_sol=jnp.array(1e-3),
    Ef_sol=jnp.array(0.0),
    alpha_ads=jnp.array(0.45),
    K0_ads=jnp.array(1.0),
    K_A_ads=jnp.array(4.5),
    K_A_des=jnp.array(0.5),
    K_B_ads=jnp.array(0.5),
    K_B_des=jnp.array(5.0),
)

newton_sol, newton_current = fd_newton.solve(params)
back_sol, backward_current = fd_backward_implicit.solve(params)
exp_sol, explicit_current = fd_explicit.solve(params)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(12, 4))

# Current Plots
args = {"alpha": 0.5}
ax1.plot(
    fd_backward_implicit.applied_potentials,
    backward_current,
    label="Implicit",
    **args,
)
ax1.plot(
    fd_explicit.applied_potentials,
    explicit_current,
    label="Explicit",
    **args,
)
ax1.plot(
    fd_newton.applied_potentials, newton_current, label="Newton", **args, linestyle="--"
)

ax1.set_title("Voltammogram")
ax1.set_ylabel(r"$J$")
ax1.set_xlabel(r"$\theta$")
ax1.xaxis.set_inverted(True)
ax1.yaxis.set_inverted(True)


# Absolute Difference
backward_diff = jnp.abs((backward_current - newton_current) / newton_current)
explicit_diff = jnp.abs((explicit_current - newton_current) / newton_current)
ax2.plot(backward_diff, label="Implicit")
ax2.plot(explicit_diff, label="Explicit")

ax2.set_title("Current")
ax2.set_xlabel("$T$")
ax2.set_ylabel("Difference")
ax2.set_yscale("log")


# Non-linear term approximation error
newton_nonlinear = newton_sol[1:-1, 2] * newton_sol[1:-1, 0]

backward_diff = jnp.abs(
    (
        back_sol[1:-1, 2] * back_sol[:-2, 0]
        + back_sol[:-2, 2] * back_sol[1:-1, 0]
        - back_sol[:-2, 2] * back_sol[:-2, 0]
        - newton_nonlinear
    )
    / newton_nonlinear
)
exp_diff = jnp.abs(
    (exp_sol[:-2, 2] * exp_sol[:-2, 0] - newton_nonlinear) / newton_nonlinear
)

ax3.plot(backward_diff, label="Implicit")
ax3.plot(exp_diff, label="Explicit")

ax3.set_title("Linear Approximation")
ax3.set_xlabel("$T$")
ax3.set_ylabel("Difference")
ax3.set_yscale("log")

handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=3)

plt.tight_layout(rect=[0, 0.08, 1, 1])
plt.show()

## Histogram

In [ ]:
rw = np.load("./data/A_RW_0.25_10.npz")
hmc = np.load("./data/A_HMC_0.25_10.npz")
true_params = AdsorptionSamplingExperiment().true_parameters


fig, axs = plt.subplots(3, 3, figsize=(10, 8))
options = {"density": True, "bins": 50, "alpha": 0.8, "histtype": "step"}
true_options = {"color": "black", "linestyle": "--"}

axs[0, 0].hist(hmc["alpha_sol"].flatten(), **options)
axs[0, 0].hist(rw["alpha_sol"].flatten(), **options)
axs[0, 0].axvline(x=true_params.alpha_ads, **true_options)
axs[0, 0].set_title(r"$\alpha^{(sol)}$")

axs[1, 0].hist(hmc["alpha_ads"].flatten(), **options)
axs[1, 0].hist(rw["alpha_ads"].flatten(), **options)
axs[1, 0].axvline(x=true_params.alpha_ads, **true_options)
axs[1, 0].set_title(r"$\alpha^{(ads)}$")

axs[0, 1].hist(hmc["K0_sol"].flatten(), **options)
axs[0, 1].hist(rw["K0_sol"].flatten(), **options)
axs[0, 1].axvline(x=true_params.K0_sol, **true_options)
axs[0, 1].set_title(r"$K_0^{(sol)}$")

axs[1, 1].hist(hmc["K0_ads"].flatten(), **options)
axs[1, 1].hist(rw["K0_ads"].flatten(), **options)
axs[1, 1].axvline(x=true_params.K0_ads, **true_options)
axs[1, 1].set_title(r"$K_0^{(ads)}$")

axs[0, 2].hist(hmc["Ef_sol"].flatten(), **options)
axs[0, 2].hist(rw["Ef_sol"].flatten(), **options)
axs[0, 2].axvline(x=true_params.Ef_sol, **true_options)
axs[0, 2].set_title(r"$E_f^{(ads)}$")

axs[1, 2].hist(hmc["K_A_ads"].flatten(), **options)
axs[1, 2].hist(rw["K_A_ads"].flatten(), **options)
axs[1, 2].axvline(x=true_params.K_A_ads, **true_options)
axs[1, 2].set_title(r"$K_{ads}^A$")

axs[2, 0].hist(hmc["K_A_des"].flatten(), **options)
axs[2, 0].hist(rw["K_A_des"].flatten(), **options)
axs[2, 0].axvline(x=true_params.K_A_des, **true_options)
axs[2, 0].set_title(r"$K_{des}^A$")

axs[2, 1].hist(hmc["K_B_ads"].flatten(), **options)
axs[2, 1].hist(rw["K_B_ads"].flatten(), **options)
axs[2, 1].axvline(x=true_params.K_B_ads, **true_options)
axs[2, 1].set_title(r"$K_{ads}^B$")

axs[2, 2].hist(hmc["K_B_des"].flatten(), **options)
axs[2, 2].hist(rw["K_B_des"].flatten(), **options)
axs[2, 2].axvline(x=true_params.K_B_des, **true_options)
axs[2, 2].set_title(r"$K_{des}^B$")

plt.tight_layout()
plt.show()

In [ ]:
rw = np.load("./data/A_RW_0.01_10.npz")

true_params = AdsorptionSamplingExperiment().true_parameters


fig, axs = plt.subplots(3, 3, figsize=(10, 8))
options = {"density": True, "bins": 50, "alpha": 0.8, "histtype": "step"}
true_options = {"color": "black", "linestyle": "--"}

# axs[0, 0].hist(hmc["alpha_sol"].flatten(), **options)
axs[0, 0].hist(rw["alpha_sol"].flatten(), **options)
axs[0, 0].axvline(x=true_params.alpha_ads, **true_options)
axs[0, 0].set_title(r"$\alpha^{(sol)}$")

# axs[1, 0].hist(hmc["alpha_ads"].flatten(), **options)
axs[1, 0].hist(rw["alpha_ads"].flatten(), **options)
axs[1, 0].axvline(x=true_params.alpha_ads, **true_options)
axs[1, 0].set_title(r"$\alpha^{(ads)}$")

# axs[0, 1].hist(hmc["K0_sol"].flatten(), **options)
axs[0, 1].hist(rw["K0_sol"].flatten(), **options)
axs[0, 1].axvline(x=true_params.K0_sol, **true_options)
axs[0, 1].set_title(r"$K_0^{(sol)}$")

# axs[1, 1].hist(hmc["K0_ads"].flatten(), **options)
axs[1, 1].hist(rw["K0_ads"].flatten(), **options)
axs[1, 1].axvline(x=true_params.K0_ads, **true_options)
axs[1, 1].set_title(r"$K_0^{(ads)}$")

# axs[0, 2].hist(hmc["Ef_sol"].flatten(), **options)
axs[0, 2].hist(rw["Ef_sol"].flatten(), **options)
axs[0, 2].axvline(x=true_params.Ef_sol, **true_options)
axs[0, 2].set_title(r"$E_f^{(ads)}$")

# axs[1, 2].hist(hmc["K_A_ads"].flatten(), **options)
axs[1, 2].hist(rw["K_A_ads"].flatten(), **options)
axs[1, 2].axvline(x=true_params.K_A_ads, **true_options)
axs[1, 2].set_title(r"$K_{ads}^A$")

# axs[2, 0].hist(hmc["K_A_des"].flatten(), **options)
axs[2, 0].hist(rw["K_A_des"].flatten(), **options)
axs[2, 0].axvline(x=true_params.K_A_des, **true_options)
axs[2, 0].set_title(r"$K_{des}^A$")

# axs[2, 1].hist(hmc["K_B_ads"].flatten(), **options)
axs[2, 1].hist(rw["K_B_ads"].flatten(), **options)
axs[2, 1].axvline(x=true_params.K_B_ads, **true_options)
axs[2, 1].set_title(r"$K_{ads}^B$")

# axs[2, 2].hist(hmc["K_B_des"].flatten(), **options)
axs[2, 2].hist(rw["K_B_des"].flatten(), **options)
axs[2, 2].axvline(x=true_params.K_B_des, **true_options)
axs[2, 2].set_title(r"$K_{des}^B$")

plt.tight_layout()
plt.show()
